In [2]:
import glob
from pathlib import Path
import pandas as pd
from scipy.stats import rankdata

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'submissions'
pick = {'exp12': 'experiment12*.csv', 'exp18': 'experiment18*.csv', 'exp25': 'experiment25*.csv'}

files = {k: sorted(OUT.glob(p)) for k, p in pick.items()}
for k, v in files.items():
    assert v, f'No file matches {pick[k]} in {OUT}'
    print(k, '->', v[0].name)

subs = {k: pd.read_csv(v[0]) for k, v in files.items()}
base = subs['exp18']
assert all((s['index'] == base['index']).all() for s in subs.values())

R = pd.DataFrame({k: rankdata(s.target) / len(s) for k, s in subs.items()})
print(R.corr(method='spearman').round(4))          # how different are they?

out = base.copy()
out['target'] = R.mean(axis=1)
out.to_csv(OUT / 'blend_exp12_18_25.csv', index=False)
print('Saved:', OUT / 'blend_exp12_18_25.csv')

exp12 -> experiment12_temporal_ensemble.csv
exp18 -> experiment18_stacking.csv
exp25 -> experiment25_pseudo_stack.csv
        exp12   exp18   exp25
exp12  1.0000  0.9922  0.9943
exp18  0.9922  1.0000  0.9888
exp25  0.9943  0.9888  1.0000
Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/blend_exp12_18_25.csv
